# Free GPU for the song site

Run this notebook on a free Colab T4 (Runtime → Change runtime type → T4 GPU), then
press **Run all**. It installs the song service, opens a tunnel and adds this card
to the pool at <https://song-pool.bpiersing.workers.dev/pool>, so the site uses it
while the notebook is open. Nothing is rented and no card is needed.

The card goes away when the session ends — that is the trade for free.
It is 16 GB, which is more than the 8 GB card at home, so songs render faster.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The song service's own packages. Colab already ships a CUDA build of torch, so
# it is left alone: replacing it with a PyPI wheel would silently drop to the CPU.
!apt-get -qq install -y ffmpeg libsndfile1 git > /dev/null
!git clone --depth 1 https://github.com/soluzka/internet.git /content/internet 2>/dev/null || true
!cd /content/internet && pip install -q -r song_service/requirements.txt --no-deps
!pip install -q flask flask-cors soundfile librosa scipy demucs transformers==4.57.6 huggingface-hub==0.36.2 hf_transfer numpy==2.4.6
print('packages in place')

In [ ]:
# ACE-Step's code and the 3.5B checkpoint (about 10 GB, into the session disk).
!cd /content/internet && pip install -q git+https://github.com/ace-step/ACE-Step.git
import os, subprocess
os.environ['HF_HOME'] = '/content/hf'
subprocess.run(['python', 'song_service/install.py', '--weights'], cwd='/content/internet', check=False)
print('weights done')

In [ ]:
# The service, on the notebook's own GPU, listening on all interfaces.
import os, subprocess, time, json, urllib.request
os.environ.update(SONG_PORT='8090', SONG_HOST='0.0.0.0', SONG_WARM='1', HF_HOME='/content/hf')
log = open('/content/song.log', 'w')
proc = subprocess.Popen(['python', 'app.py', '--role', 'song'], cwd='/content/internet/song_service',
                        stdout=log, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=3).status == 200:
            break
    except Exception:
        time.sleep(5)
print('service:', urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=10).read().decode()[:200])

In [ ]:
# A free tunnel, then the card announces itself to the pool.
import re, time, json, subprocess, urllib.request, getpass
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090',
                           '--logfile', '/content/tunnel.log'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
url = ''
for _ in range(40):
    try:
        text = open('/content/tunnel.log').read()
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if found:
            url = found.group(0); break
    except FileNotFoundError:
        pass
    time.sleep(2)
print('tunnel:', url or 'not up yet, see /content/tunnel.log')

if url:
    key = getpass.getpass('Join key (from the site owner): ')
    body = json.dumps({'url': url, 'note': 'colab t4'}).encode()
    request = urllib.request.Request('https://song-pool.bpiersing.workers.dev/join', data=body,
                                     headers={'content-type': 'application/json', 'x-join-key': key})
    print(urllib.request.urlopen(request, timeout=20).read().decode())
    print('This card is in the pool until the notebook stops. Keep the tab open.')